In [1]:
"""Maze and path finding visualizer using Python's built-in Tkinter."""

import random
import tkinter as tk
from tkinter import ttk

ROWS, COLS = 17, 25
START = (0, 0)
END = (ROWS - 1, COLS - 1)

COLORS = {
    "open": "#f0f3f8", "wall": "#293650", "start": "#19a58b",
    "end": "#f16d73", "visited": "#b9c9ff", "path": "#ffd166",
}


class MazeVisualizer:
    def __init__(self, root):
        self.root = root
        root.title("Maze & Path Finding Algorithm Visualizer")
        root.configure(bg="#f4f7fb")
        root.minsize(760, 560)
        self.walls = set()
        self.animating = False
        self.dragging = False
        self.drag_value = True
        self.buttons = {}
        self.algorithm = tk.StringVar(value="BFS")

        header = tk.Frame(root, bg="#1d2d50", padx=18, pady=14)
        header.pack(fill="x")
        tk.Label(header, text="Maze & Path Finding Algorithm Visualizer",
                 bg="#1d2d50", fg="white", font=("Segoe UI", 18, "bold")).pack(anchor="w")
        tk.Label(header, text="Design & Analysis of Algorithms · BFS and DFS",
                 bg="#1d2d50", fg="#d1daf0", font=("Segoe UI", 10)).pack(anchor="w", pady=(3, 0))

        controls = tk.Frame(root, bg="#f4f7fb", padx=14, pady=10)
        controls.pack(fill="x")
        ttk.Label(controls, text="Algorithm:").pack(side="left", padx=(0, 5))
        self.choice = ttk.Combobox(controls, textvariable=self.algorithm,
                                   values=("BFS", "DFS"), state="readonly", width=7)
        self.choice.pack(side="left", padx=(0, 10))
        self.run_button = ttk.Button(controls, text="▶ Visualize", command=self.visualize)
        self.run_button.pack(side="left", padx=3)
        self.random_button = ttk.Button(controls, text="Random maze", command=self.random_maze)
        self.random_button.pack(side="left", padx=3)
        self.clear_button = ttk.Button(controls, text="Clear walls", command=self.clear_walls)
        self.clear_button.pack(side="left", padx=3)
        self.reset_button = ttk.Button(controls, text="Reset", command=self.reset)
        self.reset_button.pack(side="left", padx=3)

        board = tk.Frame(root, bg="white", padx=12, pady=12)
        board.pack(fill="both", expand=True, padx=14, pady=(0, 8))
        for r in range(ROWS):
            board.grid_rowconfigure(r, weight=1, uniform="row")
            for c in range(COLS):
                board.grid_columnconfigure(c, weight=1, uniform="col")
                cell = (r, c)
                button = tk.Button(board, text="", bg=COLORS["open"], activebackground="#dce5f2",
                                   relief="flat", bd=0, highlightthickness=1,
                                   highlightbackground="white", padx=0, pady=0,
                                   command=lambda cell=cell: self.toggle_wall(cell))
                button.grid(row=r, column=c, sticky="nsew", padx=1, pady=1)
                button.bind("<ButtonPress-1>", lambda event, cell=cell: self.start_drag(cell))
                button.bind("<B1-Enter>", lambda event, cell=cell: self.drag_wall(cell))
                self.buttons[cell] = button
        root.bind_all("<ButtonRelease-1>", lambda event: setattr(self, "dragging", False))

        legend = tk.Frame(root, bg="#f4f7fb", padx=14)
        legend.pack(fill="x")
        for label, color in (("Start", COLORS["start"]), ("End", COLORS["end"]),
                             ("Wall", COLORS["wall"]), ("Explored", COLORS["visited"]),
                             ("Path", COLORS["path"])):
            tk.Label(legend, text=f"■ {label}", fg=color, bg="#f4f7fb",
                     font=("Segoe UI", 9, "bold")).pack(side="left", padx=(0, 12))
        self.status = tk.StringVar(value="Click or drag across cells to draw walls, then press Visualize.")
        tk.Label(root, textvariable=self.status, bg="#f4f7fb", fg="#59677e",
                 anchor="w", padx=14, pady=8).pack(fill="x")

        info = tk.Frame(root, bg="white", padx=14, pady=10)
        info.pack(fill="x", padx=14, pady=(0, 14))
        tk.Label(info, text="BFS explores by layers and finds a shortest path on this grid. DFS follows one branch at a time and may not find the shortest path.",
                 bg="white", fg="#34435d", wraplength=950, justify="left").pack(anchor="w")
        tk.Label(info, text="Both: Time O(V + E) · Space O(V)   |   V = open cells, E = neighbor connections",
                 bg="white", fg="#68758b", font=("Segoe UI", 9)).pack(anchor="w", pady=(4, 0))
        self.render()

    def render(self):
        for cell, button in self.buttons.items():
            color = COLORS["wall"] if cell in self.walls else COLORS["open"]
            text = ""
            if cell == START:
                color, text = COLORS["start"], "S"
            elif cell == END:
                color, text = COLORS["end"], "E"
            button.configure(bg=color, text=text, fg="white", font=("Segoe UI", 8, "bold"))

    def toggle_wall(self, cell):
        if self.animating or cell in (START, END):
            return
        if cell in self.walls:
            self.walls.remove(cell)
        else:
            self.walls.add(cell)
        self.buttons[cell].configure(bg=COLORS["wall"] if cell in self.walls else COLORS["open"])

    def start_drag(self, cell):
        if self.animating or cell in (START, END):
            return
        self.dragging = True
        self.drag_value = cell not in self.walls
        self.set_wall(cell, self.drag_value)
        return "break"

    def drag_wall(self, cell):
        if self.dragging:
            self.set_wall(cell, self.drag_value)

    def set_wall(self, cell, value):
        if self.animating or cell in (START, END):
            return
        if value:
            self.walls.add(cell)
        else:
            self.walls.discard(cell)
        self.buttons[cell].configure(bg=COLORS["wall"] if value else COLORS["open"])

    @staticmethod
    def neighbors(cell):
        r, c = cell
        return [(nr, nc) for nr, nc in ((r-1,c),(r,c+1),(r+1,c),(r,c-1))
                if 0 <= nr < ROWS and 0 <= nc < COLS]

    def visualize(self):
        if self.animating:
            return
        self.render()
        self.animating = True
        self.set_controls(False)
        mode = self.algorithm.get()
        seen, parent, frontier, order = {START}, {}, [START], []
        found = False
        while frontier:
            current = frontier.pop(0) if mode == "BFS" else frontier.pop()
            if current == END:
                found = True
                break
            for neighbor in self.neighbors(current):
                if neighbor not in seen and neighbor not in self.walls:
                    seen.add(neighbor)
                    parent[neighbor] = current
                    frontier.append(neighbor)
            order.append(current)
        path = []
        if found:
            current = END
            while current != START:
                path.append(current)
                current = parent[current]
            path.reverse()
        self.status.set("Searching…")
        self.animate(order, path, 0, mode, found)

    def animate(self, order, path, index, mode, found):
        if index < len(order):
            cell = order[index]
            if cell not in (START, END):
                self.buttons[cell].configure(bg=COLORS["visited"])
            delay = 5 if len(order) > 300 else 14
            self.root.after(delay, lambda: self.animate(order, path, index + 1, mode, found))
            return
        if path:
            self.animate_path(path, 0, mode, len(order))
        else:
            self.finish(mode, len(order), False, 0)

    def animate_path(self, path, index, mode, explored):
        if index < len(path):
            cell = path[index]
            if cell not in (START, END):
                self.buttons[cell].configure(bg=COLORS["path"])
            self.root.after(18, lambda: self.animate_path(path, index + 1, mode, explored))
        else:
            self.finish(mode, explored, True, len(path))

    def finish(self, mode, explored, found, length):
        if found:
            self.status.set(f"Path found! {length} steps · {explored} cells explored · {mode}.")
        else:
            self.status.set(f"No path found · {explored} cells explored · {mode}. Try clearing some walls.")
        self.animating = False
        self.set_controls(True)

    def set_controls(self, enabled):
        state = "normal" if enabled else "disabled"
        for widget in (self.choice, self.run_button, self.random_button, self.clear_button, self.reset_button):
            widget.configure(state=state)

    def random_maze(self):
        if self.animating:
            return
        self.walls = {cell for cell in self.buttons if cell not in (START, END) and random.random() < .27}
        self.render()
        self.status.set("Random maze created. Press Visualize to search.")

    def clear_walls(self):
        if self.animating:
            return
        self.walls.clear()
        self.render()
        self.status.set("Walls cleared. Draw new walls or visualize the open grid.")

    def reset(self):
        self.clear_walls()
        self.status.set("Board reset. Click or drag across cells to draw walls.")


if __name__ == "__main__":
    app = tk.Tk()
    MazeVisualizer(app)
    app.mainloop()
